# Experiment EDA

Один notebook для всіх експериментів. Він читає `cell_features.csv` і `task_features.csv` за шляхом, визначеним тим самим `experiment.toml`, що використовують команди пайплайна.

In [1]:
import csv
import os
import tomllib
from html import escape
from pathlib import Path
from statistics import fmean

from IPython.display import HTML, display

experiment_path = (
    Path(os.environ.get("EXPERIMENT_CONFIG", "experiment.toml")).expanduser().resolve()
)
experiment = tomllib.loads(experiment_path.read_text(encoding="utf-8"))
required = {"schema_version", "experiment_id", "experiment_version", "artifact_dir"}
if experiment.get("schema_version") != 1 or not required.issubset(experiment):
    raise ValueError(f"invalid experiment config: {experiment_path}")
artifact_dir = Path(experiment["artifact_dir"]).expanduser()
if not artifact_dir.is_absolute():
    artifact_dir = experiment_path.parent / artifact_dir
artifact_root = (
    artifact_dir / experiment["experiment_id"] / experiment["experiment_version"]
).resolve()
cell_path = artifact_root / "features/cell_features.csv"
task_path = artifact_root / "features/task_features.csv"


def read_csv(path):
    with path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


cell_rows = read_csv(cell_path)
task_rows = read_csv(task_path)
if not cell_rows or not task_rows:
    raise ValueError("feature tables must not be empty")
identity = {
    "experiment_id": experiment["experiment_id"],
    "experiment_version": experiment["experiment_version"],
}
for name, rows in (("cell", cell_rows), ("task", task_rows)):
    if any(any(row.get(key) != value for key, value in identity.items()) for row in rows):
        raise ValueError(f"{name} table has the wrong experiment identity")
display(
    HTML(
        f"<h2>{escape(identity['experiment_id'])} · {escape(identity['experiment_version'])}</h2>"
        f"<p>{len(task_rows)} tasks · {len(cell_rows)} cells</p>"
    )
)

In [2]:
CONDITIONS = ("NO-DOC", "OPTIONAL", "DOC-FIRST")
COLORS = {"NO-DOC": "#64748b", "OPTIONAL": "#f59e0b", "DOC-FIRST": "#2563eb"}


def number(row, key):
    value = row.get(key, "")
    return None if value == "" else float(value)


def mean(rows, key):
    values = [value for row in rows if (value := number(row, key)) is not None]
    return fmean(values) if values else None


def bars(title, values, *, digits=3):
    available = [value for _, value in values if value is not None]
    maximum = max(available, default=1) or 1
    body = []
    for label, value in values:
        width = 0 if value is None else 100 * value / maximum
        shown = "—" if value is None else f"{value:.{digits}f}"
        color = COLORS.get(label, "#0f766e")
        body.append(
            "<div style='display:grid;grid-template-columns:170px 1fr 90px;"
            "gap:12px;align-items:center;margin:8px 0'>"
            f"<strong>{escape(label)}</strong><div style='background:#e8edf4;border-radius:5px'>"
            f"<div style='width:{width:.2f}%;height:22px;background:{color};"
            "border-radius:5px'></div></div>"
            f"<code>{shown}</code></div>"
        )
    display(HTML(f"<h3>{escape(title)}</h3>{''.join(body)}"))


successful = [row for row in cell_rows if row["status"] == "succeeded"]
if {row["condition"] for row in successful} != set(CONDITIONS):
    raise ValueError("cell table must contain all three conditions")

## Результати на рівні запусків

In [3]:
METRICS = {
    "Recall@3": ("recall_at_3", 3),
    "Recall@5": ("recall_at_5", 3),
    "nDCG@3": ("ndcg_at_3", 3),
    "Returned-set F1": ("returned_set_f1", 3),
    "Provider tokens": ("provider_total_tokens", 0),
    "Elapsed seconds": ("elapsed_seconds", 1),
    "Agent steps": ("agent_step_count", 1),
    "Wiki reads": ("wiki_read_count", 2),
    "Wiki tokens": ("wiki_tokens", 0),
    "Unique wiki pages": ("unique_wiki_pages", 2),
    "Reads beyond entry": ("beyond_entry_reads", 2),
}
for title, (key, digits) in METRICS.items():
    values = [
        (condition, mean([row for row in successful if row["condition"] == condition], key))
        for condition in CONDITIONS
    ]
    bars(title, values, digits=digits)

for title, key in (
    ("Gold seen anywhere", "gold_seen_any"),
    ("Gold seen by three source actions", "gold_seen_by_3_source_actions"),
    ("Gold targeted directly", "gold_targeted_any"),
):
    bars(
        title,
        [
            (condition, mean([row for row in successful if row["condition"] == condition], key))
            for condition in CONDITIONS
        ],
    )

## Парні агрегати на рівні задач

In [4]:
task_types = {
    task_type: sum(row["task_type"] == task_type for row in task_rows)
    for task_type in sorted({row["task_type"] for row in task_rows})
}
display(
    HTML(
        "<h3>Task types</h3><ul>"
        + "".join(
            f"<li><code>{escape(key)}</code>: {value}</li>" for key, value in task_types.items()
        )
        + "</ul>"
    )
)
for title, key in (
    ("DOC-FIRST − OPTIONAL: Recall@3", "doc_first_minus_optional_recall_at_3"),
    ("DOC-FIRST − NO-DOC: Recall@3", "doc_first_minus_no_doc_recall_at_3"),
    ("OPTIONAL − NO-DOC: Recall@3", "optional_minus_no_doc_recall_at_3"),
):
    bars(title, [("task mean", mean(task_rows, key))])